In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import time

pd.set_option("display.max_rows", 999)

In [2]:
df_atac = pd.read_csv('./data/sci_CAR/ATAC_lsi.csv')

In [3]:
df_atac.head()

,Unnamed: 0,LSI_1,LSI_2,LSI_3,LSI_4,LSI_5,LSI_6,LSI_7,LSI_8,LSI_9,...,LSI_41,LSI_42,LSI_43,LSI_44,LSI_45,LSI_46,LSI_47,LSI_48,LSI_49,LSI_50
0,coRNA-RNA-plate1-001.ACGCTTCTCT,1.235860,-0.589950,-0.188422,-0.162785,0.171580,0.217535,-0.026980,-0.150939,0.162742,...,-0.134089,-0.390097,0.205378,0.171443,-0.463933,0.193463,-0.301578,-0.257534,0.204517,0.217170
1,coRNA-RNA-plate1-001.ACGTTGAATG,-1.214151,0.051744,0.034084,-0.131515,0.250671,0.096888,-0.025319,-0.084519,-0.051233,...,-0.081400,-0.279757,0.041424,-0.094950,0.052894,-0.046231,-0.044693,-0.156279,0.003671,-0.014236
2,coRNA-RNA-plate1-001.AGGACTGCGA,1.539161,-1.087436,-0.162633,-0.528799,1.287982,0.616773,-0.033681,0.058525,-0.004076,...,-0.454690,-0.329130,-0.332379,-0.314326,-0.662868,0.164304,0.413393,-0.563435,-0.139946,-0.608332
3,coRNA-RNA-plate1-001.AGGCCGGTAA,1.409260,-0.900286,-0.335751,-0.041747,1.116417,0.784632,0.231877,-0.205597,0.829931,...,-0.272118,0.745326,0.351943,-0.061394,0.174407,-0.057272,1.016228,-0.031941,-0.302164,0.452359
4,coRNA-RNA-plate1-001.CATGACTCAA,0.912757,-0.492848,-0.036632,0.184494,-0.874754,-0.220671,-0.291771,0.154704,-0.555946,...,0.724319,0.249980,-0.283260,0.348748,-0.160809,0.197030,-0.505474,0.222186,0.148584,-0.277558


In [4]:
df_rna = pd.read_csv('./data/sci_CAR/RNA_pca.csv')
df_rna.head()

,Unnamed: 0,PC_1,PC_2,PC_3,PC_4,PC_5,PC_6,PC_7,PC_8,PC_9,...,PC_41,PC_42,PC_43,PC_44,PC_45,PC_46,PC_47,PC_48,PC_49,PC_50
0,coRNA-RNA-plate1-001.ACGCTTCTCT,-7.017167,6.967923,-0.890448,-2.346056,-1.416960,1.506067,1.304570,-0.858997,-1.087285,...,0.421661,0.874643,-2.150176,0.240963,0.924930,-0.873113,-1.058278,-0.942198,-1.421278,-1.037981
1,coRNA-RNA-plate1-001.ACGTTGAATG,6.305976,-0.369384,-1.355127,-1.655408,-4.657099,2.963242,0.986717,-0.206425,2.918024,...,0.507215,0.073329,-0.326177,0.232422,0.086111,0.251889,0.328241,-0.404705,-1.132548,-0.279805
2,coRNA-RNA-plate1-001.AGGACTGCGA,5.064980,-0.600983,-0.238661,-1.565617,-4.026646,2.802538,1.202787,-0.209955,3.695962,...,-0.189905,0.207583,-0.287082,0.259737,1.045928,0.353594,-0.530509,0.338505,-0.200046,0.299909
3,coRNA-RNA-plate1-001.AGGCCGGTAA,6.167741,-0.191367,-1.118713,-1.460923,-4.526852,2.360865,0.912821,-0.240274,2.319247,...,0.285172,0.642185,-0.307905,-1.238700,-0.797882,1.270203,0.263561,-0.023996,-1.078793,0.111626
4,coRNA-RNA-plate1-001.CATGACTCAA,-17.017403,18.867785,0.207563,-1.291055,1.401379,3.686704,-3.243650,1.398786,1.698218,...,2.680566,-0.044568,0.220771,-0.641249,0.782806,-0.290419,-2.359336,1.719087,-2.390894,-1.876154


In [5]:

def process_data(multi_omics_data: dict[str, Path], cell_key: str, dataname: str, prefix: Path, fast_edge=True):

    def get_omics_id_from_gene(gene_name: str):
        for id, omics_name in enumerate(multi_omics_data.keys()):
            if gene_name.endswith(omics_name):
                return id
        raise ValueError(gene_name + ' has no omics!')

    num_omics = len(multi_omics_data)
    print('Begin to load multi omics data', multi_omics_data)
    begin = time.time()
    multi_omics_df = {
        name: pd.read_csv(path, index_col=cell_key).sort_index() for name, path in multi_omics_data.items()
    }
    print('Load data done, time:', time.time() - begin)

    # Rename feature columns.
    multi_omics_df = {
        name: df.rename({col: col.strip() + "_" + name for col in df.columns}, axis=1)
          for name, df in multi_omics_df.items()
    }
    print('Rename columns')
    for name, df in multi_omics_df.items():
        print('Name', name, 'Columns', df.columns)

    print('Union features from multi-omics')
    # Join the dataframes by cell key.
    df = pd.concat(list(multi_omics_df.values()), axis=1)

    # Remove all-NaNs columns
    df.dropna(axis=1, how='all', inplace=True)
    num_cells = len(df)
    num_genes = len(df.columns)
    num_nodes = num_cells + num_genes + num_omics

    print(f'{num_omics=}, {num_cells=}, {num_genes=}, {num_nodes=}')

    def add_node_type(nodes, node_type):
        return [(node, node_type) for node in nodes]

    omics_names = ['Omics_' + name for name in multi_omics_data.keys()]
    cell_names = df.index.values.tolist()
    gene_names = list(df.columns)
    node_names = add_node_type(omics_names, 0) + add_node_type(gene_names, 1) + \
        add_node_type(cell_names, 2)
    df_nodes = pd.DataFrame(node_names, columns=['Name', 'Type'])

    # Node offsets
    omics_offset = 0
    gene_offset = num_omics
    cell_offset = num_omics + num_genes

    def construct_hyper_edge():
        hyper_edges = []
        for cell_id in range(num_cells):
            for gene_id in range(num_genes):
                weight = df.iloc[cell_id, gene_id]
                if np.isnan(weight):
                    continue

                omics_id = get_omics_id_from_gene(gene_names[gene_id])
                omics_node = omics_id + omics_offset
                gene_node = gene_id + gene_offset
                cell_node = cell_id + cell_offset
                edge = dict(Omics=omics_node, Gene=gene_node,
                            Cell=cell_node, Weight=weight)
                hyper_edges.append(edge)

        df_edges =  pd.DataFrame.from_records(hyper_edges)
        return df_edges

    def construct_hyper_edge_fast():

        # 0. 预计算 gene_id → omics_id
        gene2omics = pd.Series(
            [get_omics_id_from_gene(g) for g in df.columns],
            index=pd.RangeIndex(len(df.columns))   # 0,1,2,...
        )

        # 1. 把列名换成整数，再 stack
        df_int_cols = df.set_axis(range(df.shape[1]), axis=1)   # 关键一步
        w = df_int_cols.to_numpy()
        cell_id, gene_id = np.nonzero(~np.isnan(w))   # 非 NaN 的坐标
        weight = w[cell_id, gene_id]
        long = pd.DataFrame({'cell_id': cell_id,
                     'gene_id': gene_id,
                     'Weight': weight})
        print('Long format data')

        # ---- 2. 三列 node id 向量化 ----
        long['Omics']  = gene2omics[long['gene_id']].values + omics_offset
        long['Gene']   = long['gene_id'].values + gene_offset
        long['Cell']   = long['cell_id'].values + cell_offset

        # ---- 3. 只要这四列 ----
        hyper_edges = long[['Omics', 'Gene', 'Cell', 'Weight']]
        return hyper_edges

    begin = time.time()
    print('Constructing hyper edges...')
    df_edges = construct_hyper_edge_fast() if fast_edge else construct_hyper_edge()
    print('Construct edges done, time:', time.time() - begin)

    savedir = Path(prefix) / dataname
    savedir.mkdir(parents=True, exist_ok=True)
    node_file = savedir / 'nodes.csv'
    df_nodes.to_csv(node_file, index=True)
    print('Save node file', node_file)

    edge_file = savedir / 'edges.csv'
    df_edges.to_csv(edge_file, index=False)
    print('Save edge file', edge_file)

    metadata = dict(
        num_cells=num_cells, num_genes=num_genes, num_omics=num_omics,
        num_nodes=num_nodes, num_edges=len(df_edges),
        omics_offset=omics_offset, gene_offset=gene_offset, cell_offset=cell_offset,
        raw_data=multi_omics_data,
        cell_key=cell_key,
    )

    metadata_file = savedir / 'metadata.json'
    json.dump(metadata, metadata_file.open('w'), indent=4)
    print('Save metadata', metadata_file)

In [6]:
process_data({
        'expr': './data/sc_GEM/expression_data.csv',
        'methy': './data/sc_GEM/methylation_data.csv',
    }, cell_key='Cell ID', dataname='sc_GEM', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/sc_GEM/expression_data.csv', 'methy': './data/sc_GEM/methylation_data.csv'}
Load data done, time: 0.013839960098266602
Rename columns
Name expr Columns Index(['ADAM33_expr', 'AFP_expr', 'AK123759_expr', 'ALDH3A1_expr', 'BMP7_expr',
       'CDH1_expr', 'CDH22_expr', 'CDX2_expr', 'CER1_expr', 'CHL1_expr',
       'COL20A1_expr', 'COL23A1_expr', 'CYBRD1_expr', 'DAZL_expr',
       'DNMT3B_expr', 'DNMT3L_expr', 'DPPA3_expr', 'EPHB3_expr', 'FGF4_expr',
       'FOSL1_expr', 'FOXD2_expr', 'HAND1_expr', 'HLX-AS1_expr', 'IGF2_expr',
       'JARID2_expr', 'KCNQ2_expr', 'LEFTY_expr', 'LTBR_expr', 'LUM_expr',
       'LY86-AS1_expr', 'MMP9_expr', 'MYC_expr', 'NANOG_expr', 'NESTIN_expr',
       'NFATC1_expr', 'NFIX_expr', 'NKX2-5_expr', 'NXRA8_expr', 'OCT4_expr',
       'OTX2_expr', 'PIWIL1_expr', 'PLEKHH3_expr', 'PRDM14_expr', 'SALL4_expr',
       'SOX2_expr', 'STON2P1_expr', 'SULT1A1_expr', 'TBX3_expr', 'TFAP2A_expr',
       'TFCP2L1_expr', 'TGFBR2_exp

In [7]:
process_data({
        'expr': './data/PEA_STA/expression_data.csv',
        'protein': './data/PEA_STA/protein_data.csv',
    }, cell_key='Cells ', dataname='PEA_STA', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/PEA_STA/expression_data.csv', 'protein': './data/PEA_STA/protein_data.csv'}
Load data done, time: 0.016832828521728516
Rename columns
Name expr Columns Index(['AKT1_expr', 'APC_expr', 'APP_expr', 'AURKB_expr', 'AXIN1_expr',
       'AXIN2_expr', 'BAX_expr', 'BCL2_expr', 'BCL2L1_expr', 'BCL6_expr',
       ...
       'THY1_expr', 'TNFRSF10B_expr', 'TNFRSF1A_expr', 'TP53_expr',
       'TRADD_expr', 'TUBB3_expr', 'TWSG1_expr', 'VEGFA_expr', 'VIM_expr',
       'XIAP_expr'],
      dtype='object', length=141)
Name protein Columns Index(['AKT1_protein', 'APC_protein', 'APP_protein', 'AURKB_protein',
       'AXIN1_protein', 'AXIN2_protein', 'BAX_protein', 'BCL2_protein',
       'BCL2L1_protein', 'BCL6_protein',
       ...
       'THY1_protein', 'TNFRSF10B_protein', 'TNFRSF1A_protein', 'TP53_protein',
       'TRADD_protein', 'TUBB3_protein', 'TWSG1_protein', 'VEGFA_protein',
       'VIM_protein', 'XIAP_protein'],
      dtype='object', length=141)
Un

In [8]:
process_data({
        'atac': './data/sci_CAR/ATAC_lsi.csv',
        'rna': './data/sci_CAR/RNA_pca.csv',
    }, cell_key='Unnamed: 0', dataname='sci_CAR', prefix=Path('./hypergraph'))

Begin to load multi omics data {'atac': './data/sci_CAR/ATAC_lsi.csv', 'rna': './data/sci_CAR/RNA_pca.csv'}


Load data done, time: 0.3490123748779297
Rename columns
Name atac Columns Index(['LSI_1_atac', 'LSI_2_atac', 'LSI_3_atac', 'LSI_4_atac', 'LSI_5_atac',
       'LSI_6_atac', 'LSI_7_atac', 'LSI_8_atac', 'LSI_9_atac', 'LSI_10_atac',
       'LSI_11_atac', 'LSI_12_atac', 'LSI_13_atac', 'LSI_14_atac',
       'LSI_15_atac', 'LSI_16_atac', 'LSI_17_atac', 'LSI_18_atac',
       'LSI_19_atac', 'LSI_20_atac', 'LSI_21_atac', 'LSI_22_atac',
       'LSI_23_atac', 'LSI_24_atac', 'LSI_25_atac', 'LSI_26_atac',
       'LSI_27_atac', 'LSI_28_atac', 'LSI_29_atac', 'LSI_30_atac',
       'LSI_31_atac', 'LSI_32_atac', 'LSI_33_atac', 'LSI_34_atac',
       'LSI_35_atac', 'LSI_36_atac', 'LSI_37_atac', 'LSI_38_atac',
       'LSI_39_atac', 'LSI_40_atac', 'LSI_41_atac', 'LSI_42_atac',
       'LSI_43_atac', 'LSI_44_atac', 'LSI_45_atac', 'LSI_46_atac',
       'LSI_47_atac', 'LSI_48_atac', 'LSI_49_atac', 'LSI_50_atac'],
      dtype='object')
Name rna Columns Index(['PC_1_rna', 'PC_2_rna', 'PC_3_rna', 'PC_4_rna', 'PC_5_

In [9]:
process_data({
        'expr': './data/scNMT/expression_data_300.csv',
        'rna': './data/scNMT/RNA_pca.csv',
    }, cell_key='Unnamed: 0', dataname='scNMT', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/scNMT/expression_data_300.csv', 'rna': './data/scNMT/RNA_pca.csv'}


ValueError: Index Unnamed: 0 invalid

In [ ]:
process_data({
        'expr': './data/SCoPE2/expression_data.csv',
        'protein': './data/SCoPE2/protein_data.csv',
    }, cell_key='Unnamed: 0', dataname='SCoPE2', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/SCoPE2/expression_data.csv', 'protein': './data/SCoPE2/protein_data.csv'}
Load data done, time: 0.8750507831573486
Rename columns
Name expr Columns Index(['A0A075B6H9_expr', 'A0A0B4J1V0_expr', 'A0A0B4J237_expr',
       'A0A1B0GTH6_expr', 'A0A1B0GUA6_expr', 'A0A1B0GUU1_expr',
       'A0A1B0GUW6_expr', 'A0AVF1_expr', 'A0AVT1_expr', 'A0M8Q6_expr',
       ...
       'Q9Y6F8_expr', 'Q9Y6H5_expr', 'Q9Y6I0_expr', 'Q9Y6M7_expr',
       'Q9Y6N5_expr', 'Q9Y6R9_expr', 'Q9Y6U7_expr', 'Q9Y6W6_expr',
       'Q9Y6X6_expr', 'Q9Y6Z7_expr'],
      dtype='object', length=3042)
Name protein Columns Index(['A0A075B6H9_protein', 'A0A0B4J1V0_protein', 'A0A0B4J237_protein',
       'A0A1B0GTH6_protein', 'A0A1B0GUA6_protein', 'A0A1B0GUU1_protein',
       'A0A1B0GUW6_protein', 'A0AVF1_protein', 'A0AVT1_protein',
       'A0M8Q6_protein',
       ...
       'Q9Y6F8_protein', 'Q9Y6H5_protein', 'Q9Y6I0_protein', 'Q9Y6M7_protein',
       'Q9Y6N5_protein', 'Q9Y6R9_protein

In [ ]:
process_data({
        'expr': './data/CITE_seq/expression_data.csv',
        'protein': './data/CITE_seq/protein_data.csv',
    }, cell_key='cell ID', dataname='CITE_seq', prefix=Path('./hypergraph'))

Begin to load multi omics data {'expr': './data/CITE_seq/expression_data.csv', 'protein': './data/CITE_seq/protein_data.csv'}
Load data done, time: 86.1108238697052
Rename columns
Name expr Columns Index(['ERCC_ERCC-00104_expr', 'HUMAN_A1BG_expr', 'HUMAN_A1BG-AS1_expr',
       'HUMAN_A1CF_expr', 'HUMAN_A2M_expr', 'HUMAN_A2M-AS1_expr',
       'HUMAN_A2ML1_expr', 'HUMAN_A4GALT_expr', 'HUMAN_A4GNT_expr',
       'HUMAN_AAAS_expr',
       ...
       'MOUSE_mt-Ti_expr', 'MOUSE_mt-Tl1_expr', 'MOUSE_mt-Tm_expr',
       'MOUSE_mt-Tp_expr', 'MOUSE_mt-Tq_expr', 'MOUSE_mt-Tt_expr',
       'MOUSE_mt-Tw_expr', 'MOUSE_n-R5s200_expr', 'MOUSE_n-R5s25_expr',
       'MOUSE_n-R5s31_expr'],
      dtype='object', length=36280)
Name protein Columns Index(['CD3_protein', 'CD4_protein', 'CD8_protein', 'CD45RA_protein',
       'CD56_protein', 'CD16_protein', 'CD10_protein', 'CD11c_protein',
       'CD14_protein', 'CD19_protein', 'CD34_protein', 'CCR5_protein',
       'CCR7_protein'],
      dtype='object')
Union

: 